# FastAPI Endpoints Test Notebook

This notebook tests the following FastAPI endpoints:
- `/api/v1/documents/upload` - Upload documents
- `/api/v1/documents/{id}/download` - Download documents
- `/api/v1/search/` - Search documents
- `/api/v1/search/semantic` - Semantic search
- `/api/v1/analytics/overview` - Analytics
- `/api/v1/tags/` - List tags

Test files are located in: `/Users/glennmossy/dpg-ai-projects/claude_document_mcp_server/testfiles`


In [1]:
import requests
import json
from pathlib import Path
from typing import Dict, Any
import os

# API base URL - adjust if needed
BASE_URL = "http://localhost/api/v1"  # Using nginx reverse proxy
# BASE_URL = "http://localhost:8000/api/v1"  # Direct backend access

# Test files directory
TESTFILES_DIR = Path("/Users/glennmossy/dpg-ai-projects/claude_document_mcp_server/testfiles")

print(f"API Base URL: {BASE_URL}")
print(f"Test Files Directory: {TESTFILES_DIR}")
print(f"Test Files Directory Exists: {TESTFILES_DIR.exists()}")


API Base URL: http://localhost/api/v1
Test Files Directory: /Users/glennmossy/dpg-ai-projects/claude_document_mcp_server/testfiles
Test Files Directory Exists: True


## 1. Health Check


In [2]:
# Health check
response = requests.get(f"{BASE_URL}/healthz")
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")


Status: 200
Response: {'status': 'ok'}


## 2. Upload Document Test


In [3]:
# List available test files
test_files = list(TESTFILES_DIR.glob("*")) if TESTFILES_DIR.exists() else []
print(f"Available test files: {len(test_files)}")
for f in test_files[:10]:  # Show first 10
    print(f"  - {f.name} ({f.stat().st_size} bytes)")

# Upload a test file
uploaded_documents = []

if test_files:
    test_file = test_files[0]  # Use first available file
    
    with open(test_file, 'rb') as f:
        files = {'file': (test_file.name, f, 'application/octet-stream')}
        data = {
            'title': f'Test Document - {test_file.name}',
            'tags': json.dumps(['test', 'notebook', 'api']),
            'status': 'draft',
            'metadata': json.dumps({
                'source': 'test_notebook',
                'category': 'testing'
            })
        }
        
        response = requests.post(f"{BASE_URL}/documents/upload", files=files, data=data)
        
        print(f"\nUpload Status: {response.status_code}")
        if response.status_code == 200:
            result = response.json()
            print(f"\nUpload Response:")
            print(json.dumps(result, indent=2))
            
            # Store document ID for later tests
            if 'document_id' in result:
                uploaded_documents.append({
                    'document_id': result['document_id'],
                    'title': result.get('title', ''),
                    'version': result.get('version', 1)
                })
                print(f"\n✅ Document uploaded successfully!")
                print(f"   Document ID: {result['document_id']}")
                print(f"   Version: {result.get('version', 1)}")
        else:
            print(f"❌ Upload failed: {response.text}")
else:
    print("⚠️  No test files found in testfiles directory")


Available test files: 24
  - Excel_Test_1_5pages_2470.xlsx (25752 bytes)
  - PDF_Test_3_4pages_5077.pdf (9449 bytes)
  - Text_Test_1_2pages_2688.txt (16029 bytes)
  - Word_Test_2_4pages_4571.docx (41461 bytes)
  - CUE_Test_2_3pages_7965.cue (2490 bytes)
  - CUE_Test_1_4pages_4682.cue (2964 bytes)
  - Text_Test_2_3pages_8953.txt (21070 bytes)
  - Text_Test_2_5pages_3978.txt (31768 bytes)
  - CUE_Test_3_3pages_1468.cue (2513 bytes)
  - Excel_Test_3_2pages_3281.xlsx (14002 bytes)

Upload Status: 200

Upload Response:
{
  "success": true,
  "document_id": "doc_38fd967d69e4",
  "title": "Test Document - Excel_Test_1_5pages_2470.xlsx",
  "status": "draft",
  "created_at": "2025-12-01T15:25:37.819073+00:00",
  "size": 29,
  "tags": [
    "test",
    "notebook",
    "api"
  ],
  "version": 1,
  "message": "Document 'Test Document - Excel_Test_1_5pages_2470.xlsx' created successfully with ID doc_38fd967d69e4",
  "binary": {
    "filename": "Excel_Test_1_5pages_2470.xlsx",
    "mime_type": "appl

## 3. Download Document Test


In [4]:
# Download the uploaded document
if uploaded_documents:
    doc = uploaded_documents[0]
    doc_id = doc['document_id']
    
    response = requests.get(f"{BASE_URL}/documents/{doc_id}/download")
    
    print(f"Download Status: {response.status_code}")
    
    if response.status_code == 200:
        # Get filename from Content-Disposition header
        content_disposition = response.headers.get('Content-Disposition', '')
        print(f"Content-Disposition: {content_disposition}")
        print(f"Content-Type: {response.headers.get('Content-Type', 'unknown')}")
        print(f"Content-Length: {len(response.content)} bytes")
        
        # Save downloaded file
        output_file = TESTFILES_DIR / f"downloaded_{doc_id}.bin"
        with open(output_file, 'wb') as f:
            f.write(response.content)
        
        print(f"\n✅ Document downloaded successfully!")
        print(f"   Saved to: {output_file}")
    else:
        print(f"❌ Download failed: {response.text}")
else:
    print("⚠️  No documents uploaded yet. Run the upload test first.")


Download Status: 200
Content-Disposition: attachment; filename="Excel_Test_1_5pages_2470.xlsx"
Content-Type: application/octet-stream
Content-Length: 25752 bytes

✅ Document downloaded successfully!
   Saved to: /Users/glennmossy/dpg-ai-projects/claude_document_mcp_server/testfiles/downloaded_doc_38fd967d69e4.bin


## 4. Search Documents Test


In [5]:
# Search for documents
search_query = "test"
limit = 10

response = requests.get(f"{BASE_URL}/search/", params={'q': search_query, 'limit': limit})

print(f"Search Status: {response.status_code}")

if response.status_code == 200:
    result = response.json()
    print(f"\nSearch Results for '{search_query}':")
    print(json.dumps(result, indent=2))
    
    results = result.get('results', [])
    print(f"\n✅ Found {len(results)} documents")
    
    for doc in results[:5]:  # Show first 5
        print(f"  - {doc.get('title', 'N/A')} (ID: {doc.get('document_id', 'N/A')})")
else:
    print(f"❌ Search failed: {response.text}")


Search Status: 200

Search Results for 'test':
{
  "results": [
    {
      "document_id": "doc_38fd967d69e4",
      "title": "Test Document - Excel_Test_1_5pages_2470.xlsx",
      "status": "draft",
      "tags": [
        "test",
        "notebook",
        "api"
      ],
      "created_at": "2025-12-01T15:25:37.819073+00:00",
      "updated_at": "2025-12-01T15:25:37.819073+00:00",
      "size": 29
    },
    {
      "document_id": "doc_e66f663a8da4",
      "title": "My Document Title Test",
      "status": "draft",
      "tags": [
        "Joint Test"
      ],
      "created_at": "2025-11-26T22:48:34.033719+00:00",
      "updated_at": "2025-11-26T22:48:34.033719+00:00",
      "size": 81
    },
    {
      "document_id": "doc_2bd2ca96fb97",
      "title": "Updated API Test Document",
      "status": "archived",
      "tags": [
        "bulk-tagged",
        "updated",
        "api",
        "test",
        "integration"
      ],
      "created_at": "2025-11-26T22:24:47.497527+00:00",

## 5. Semantic Search Test


In [6]:
# Semantic search
semantic_query = "test document"
limit = 10

payload = {
    'query': semantic_query,
    'limit': limit
}

response = requests.post(f"{BASE_URL}/search/semantic", json=payload)

print(f"Semantic Search Status: {response.status_code}")

if response.status_code == 200:
    result = response.json()
    print(f"\nSemantic Search Results for '{semantic_query}':")
    print(json.dumps(result, indent=2))
    
    results = result.get('results', [])
    print(f"\n✅ Found {len(results)} documents")
else:
    print(f"❌ Semantic search failed: {response.text}")
    print(f"Note: Semantic search may not be fully implemented")


Semantic Search Status: 200

Semantic Search Results for 'test document':
{
  "results": [
    {
      "document_id": "doc_cef1f711b090",
      "title": "API Test Document",
      "status": "archived",
      "tags": [
        "test",
        "api",
        "integration"
      ],
      "snippet": "<b>test</b>_upload_<b>document</b>.txt"
    },
    {
      "document_id": "doc_9bd21215b9a2",
      "title": "AnnualEnergyData_20251021_041609_3894845",
      "status": "draft",
      "tags": [],
      "snippet": "<b>AnnualEnergyData</b>_20251021_<b>041609</b>_3894845.xlsx"
    },
    {
      "document_id": "doc_3e7b439c0eee",
      "title": "Cleaned_Teams_Transcript_Summary",
      "status": "draft",
      "tags": [],
      "snippet": "<b>Cleaned</b>_Teams_<b>Transcript</b>_Summary.docx"
    },
    {
      "document_id": "doc_923196a1e932",
      "title": "AnnualEnergyData_20251021_041609_3894845",
      "status": "draft",
      "tags": [],
      "snippet": "<b>AnnualEnergyData</b>_20251021_<

## 6. Analytics Overview Test


In [7]:
# Get analytics overview
response = requests.get(f"{BASE_URL}/analytics/overview")

print(f"Analytics Status: {response.status_code}")

if response.status_code == 200:
    result = response.json()
    print(f"\nAnalytics Overview:")
    print(json.dumps(result, indent=2))
    
    totals = result.get('totals', {})
    print(f"\n✅ Analytics retrieved successfully")
    print(f"   Totals: {totals}")
else:
    print(f"❌ Analytics request failed: {response.text}")


Analytics Status: 200

Analytics Overview:
{
  "totals": {}
}

✅ Analytics retrieved successfully
   Totals: {}


## 7. List Tags Test


In [8]:
# List all tags
response = requests.get(f"{BASE_URL}/tags/", params={
    'sort_by_count': True,
    'min_count': 1
})

print(f"Tags Status: {response.status_code}")

if response.status_code == 200:
    result = response.json()
    print(f"\nTags List:")
    print(json.dumps(result, indent=2))
    
    tags = result.get('tags', [])
    total = result.get('total', 0)
    
    print(f"\n✅ Found {total} unique tags")
    
    for tag_info in tags[:10]:  # Show first 10
        tag_name = tag_info.get('tag', 'N/A')
        count = tag_info.get('count', 0)
        print(f"  - {tag_name}: {count} documents")
else:
    print(f"❌ Tags request failed: {response.text}")


Tags Status: 200

Tags List:
{
  "tags": [
    {
      "tag": "test",
      "count": 8
    },
    {
      "tag": "api",
      "count": 8
    },
    {
      "tag": "integration",
      "count": 7
    },
    {
      "tag": "bulk-tagged",
      "count": 6
    },
    {
      "tag": "updated",
      "count": 6
    },
    {
      "tag": "notebook",
      "count": 1
    },
    {
      "tag": "GenAI",
      "count": 1
    },
    {
      "tag": "Joint Test",
      "count": 1
    }
  ],
  "total": 8
}

✅ Found 8 unique tags
  - test: 8 documents
  - api: 8 documents
  - integration: 7 documents
  - bulk-tagged: 6 documents
  - updated: 6 documents
  - notebook: 1 documents
  - GenAI: 1 documents
  - Joint Test: 1 documents


## 8. Test Summary


In [9]:
# Summary of uploaded documents
print("\n" + "="*50)
print("TEST SUMMARY")
print("="*50)

print(f"\nUploaded Documents: {len(uploaded_documents)}")
for doc in uploaded_documents:
    print(f"  - {doc['title']} (ID: {doc['document_id']}, Version: {doc['version']})")

print(f"\n✅ All endpoint tests completed!")
print(f"\nTest files directory: {TESTFILES_DIR}")
print(f"API Base URL: {BASE_URL}")



TEST SUMMARY

Uploaded Documents: 1
  - Test Document - Excel_Test_1_5pages_2470.xlsx (ID: doc_38fd967d69e4, Version: 1)

✅ All endpoint tests completed!

Test files directory: /Users/glennmossy/dpg-ai-projects/claude_document_mcp_server/testfiles
API Base URL: http://localhost/api/v1
